# Sketch 3.5 - standalone, portable notebook

Self-contained extract of **sketch_3_5_scatter** from the combined *9 sketches*
notebook. The sketch code is unchanged; the shared setup it needs
(imports, data loading) is included below. Chart data is **inlined** (no external
`altair-data` files) so it renders on Linux, macOS and Windows alike.

## Shared setup

In [ ]:
import altair as alt
import pandas as pd
import numpy as np
import geopandas as gpd  # conda install -c conda-forge geopandas
import json

# Inline the chart data so every HTML output is SELF-CONTAINED and renders on
# all platforms (incl. macOS): external altair-data-*.json URL files are not
# fetched by many notebook frontends, which left the charts blank on Mac. The
# search sketches (1.13 / 2.4 / 2.6) restrict their name universe below so the
# inlined tables stay small.
alt.data_transformers.enable('default')
alt.data_transformers.disable_max_rows()
pass

In [ ]:
# Memory-lean load: another heavy job is running on this machine (~2.5 GB free),
# so we read the CSV with categorical dtypes (15k name strings stored once), derive
# the numeric year from the 122 'annais' categories instead of 3.5M strings, then
# relax categories back to plain objects (the strings stay shared) so every later
# groupby keeps its normal semantics.
raw = pd.read_csv('dpt2020.csv', sep=';',
                  dtype={'sexe': 'int8', 'preusuel': 'category',
                         'annais': 'category', 'dpt': 'category', 'nombre': 'int32'})
cat_years = pd.to_numeric(pd.Index(raw['annais'].cat.categories), errors='coerce').astype('float32')
raw['year'] = np.asarray(cat_years)[raw['annais'].cat.codes]
raw.drop(columns=['annais'], inplace=True)
raw['decade'] = (raw['year'] // 10 * 10)
raw['preusuel'] = raw['preusuel'].astype(object)  # shared strings, plain groupbys
raw['dpt'] = raw['dpt'].astype(object)

records = raw[raw.dpt != 'XX'].copy()
named = records[records.preusuel != '_PRENOMS_RARES'].copy()

dept_total = records.groupby('dpt', as_index=False)['nombre'].sum().rename(
    columns={'nombre': 'dept_total'})

ALL_NAMES = sorted(named.preusuel.unique().tolist())  # full dropdown option list
DECADES = list(range(1900, 2021, 10))
# Shared period options for charts that need an all-years aggregate (2.4).
DECADE_OPTS = [0] + DECADES
DECADE_LABELS = ['All years'] + [str(d) for d in DECADES]
SEX_OPTIONS = ['Mixed', 'Boys', 'Girls']
SEX_LABELS = ['mixed', 'boys', 'girls']

import gc
del raw
gc.collect()

print(f'{len(named):,} named rows; {len(ALL_NAMES):,} distinct names.')
named.sample(3)


## The sketch

### Sketch 3.5 - Popularity among boys vs among girls (decade slider)

x = the name's share of all male births that decade (‰), y = its share of all
female births (‰), log axes, `y = x` parity line (= unisex). Decade slider
1900-2020 animates; size = total births (√ scale).


In [ ]:
g3 = (named.dropna(subset=['decade'])
      .groupby(['decade', 'preusuel', 'sexe'])['nombre'].sum()
      .unstack(fill_value=0).rename(columns={1: 'male', 2: 'female'}).reset_index())
tot3 = (named.dropna(subset=['decade']).groupby(['decade', 'sexe'])['nombre'].sum()
        .unstack(fill_value=0).rename(columns={1: 'tmale', 2: 'tfemale'}).reset_index())
g3 = g3.merge(tot3, on='decade')
g3 = g3[(g3.male > 0) & (g3.female > 0)].copy()
g3['male_pm'] = (1000 * g3.male / g3.tmale).round(4)
g3['female_pm'] = (1000 * g3.female / g3.tfemale).round(4)
g3['total'] = g3.male + g3.female
g3['male_share'] = (g3.male / g3.total).round(4)
g3['rank'] = g3.groupby('decade')['total'].rank(ascending=False, method='first')
g3['decade'] = g3['decade'].astype(int)

n_lab = alt.param(name='n_lab', value=8,
                  bind=alt.binding_range(min=0, max=200, step=1, name='Labelled names  '))
decade_sel = alt.selection_point(fields=['decade'], value=[{'decade': 2000}],
                                 bind=alt.binding_range(min=1900, max=2020, step=10, name='Decade  '))
lo = float(g3[['male_pm', 'female_pm']].min().min())
hi = float(g3[['male_pm', 'female_pm']].max().max())
parity = alt.Chart(pd.DataFrame({'v': [lo, hi]})).mark_line(
    color='#444', strokeDash=[6, 4]).encode(x='v:Q', y='v:Q')
pts = alt.Chart(g3).mark_circle(opacity=0.7, stroke='#888', strokeWidth=0.4).encode(
    x=alt.X('male_pm:Q', scale=alt.Scale(type='log'), title='Share of all BOYS (per 1000, log)'),
    y=alt.Y('female_pm:Q', scale=alt.Scale(type='log'), title='Share of all GIRLS (per 1000, log)'),
    size=alt.Size('total:Q', title='Total births', scale=alt.Scale(type='sqrt', range=[15, 1100])),
    color=alt.Color('male_share:Q', title='Male share', scale=alt.Scale(scheme='redblue', domain=[0, 1])),
    tooltip=[alt.Tooltip('preusuel:N', title='Name'), alt.Tooltip('decade:Q', title='Decade', format='d'),
             alt.Tooltip('male:Q', title='Boys', format=','), alt.Tooltip('female:Q', title='Girls', format=',')],
).transform_filter(decade_sel)
labels = alt.Chart(g3).mark_text(align='left', dx=6, fontSize=9).encode(
    x='male_pm:Q', y='female_pm:Q', text='preusuel:N').transform_filter(
    decade_sel).transform_filter('datum.rank <= n_lab')
sketch_3_5 = (parity + pts + labels).add_params(decade_sel, n_lab).properties(
    width=600, height=560,
    title='3.5 - Popularity among boys vs girls (parity line = unisex; drag the decade)')

sketch_3_5.save('sketch_3_5_scatter.png', ppi=120)
sketch_3_5